# 第 3 章互动实验：结构元素如何改变形状

实验使用合成二值图和 OpenCV 基线，只用于观察，不提供 `3-1_手写二值形态学.py` 的循环实现。重点关注前景像素数、孔洞、间隙和错误粘连。

In [ ]:
%matplotlib inline
import importlib.util
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, IntSlider

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Heiti SC', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError('请在课程目录内启动 Jupyter')


CHAPTER = find_chapter('03_形态学')


def load_homework(filename):
    """读入本章的一个作业文件。每次调用都重新读，改完作业后重新运行单元就能看到新结果。
    设置环境变量 COURSE_IMPL_DIR 可以换成另一份实现（维护者验证用）。"""
    override = os.environ.get('COURSE_IMPL_DIR')
    base = CHAPTER
    if override:
        base = Path(override) / CHAPTER.name
        if not base.is_dir():                      # 相对路径：按课程根目录再找一次
            base = CHAPTER.parent / override / CHAPTER.name
    spec = importlib.util.spec_from_file_location('homework_' + filename[:3].replace('-', '_'), base / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        result = fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f'先完成作业里的 {fn.__name__}：{exc}')
        return None
    if result is None:
        print(f'先完成作业里的 {fn.__name__}（现在返回的是 None）')
    return result


binary = np.zeros((120, 180), dtype=np.uint8)
binary[25:90, 20:65] = 255
binary[43:70, 35:50] = 0                 # 孔洞
binary[35:80, 82:120] = 255
binary[35:80, 125:162] = 255             # 5 像素间隙
rng = np.random.default_rng(2026)
ys = rng.integers(0, binary.shape[0], 35)
xs = rng.integers(0, binary.shape[1], 35)
binary[ys, xs] = 255

In [ ]:
OPERATIONS = {
    '腐蚀': cv2.MORPH_ERODE, '膨胀': cv2.MORPH_DILATE,
    '开运算': cv2.MORPH_OPEN, '闭运算': cv2.MORPH_CLOSE,
}

def show_morphology(operation='闭运算', width=7, height=3):
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (width, height))
    code = OPERATIONS[operation]
    if code == cv2.MORPH_ERODE:
        result = cv2.erode(binary, kernel)
    elif code == cv2.MORPH_DILATE:
        result = cv2.dilate(binary, kernel)
    else:
        result = cv2.morphologyEx(binary, code, kernel)
    n0, _, _, _ = cv2.connectedComponentsWithStats(binary, 8)
    n1, _, _, _ = cv2.connectedComponentsWithStats(result, 8)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(binary, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'原图：前景 {np.count_nonzero(binary)}，区域 {n0-1}')
    axes[1].imshow(result, cmap='gray', vmin=0, vmax=255)
    axes[1].set_title(f'{operation} {width}×{height}：前景 {np.count_nonzero(result)}，区域 {n1-1}')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_morphology,
    operation=Dropdown(options=list(OPERATIONS), value='闭运算'),
    width=IntSlider(min=1, max=21, step=2, value=7),
    height=IntSlider(min=1, max=15, step=2, value=3),
);

## 讲义新增内容的三个实验

下面三个实验对应讲义 3.2、3.3、3.4 节新增的实测内容。每个实验先写下预测，再运行。

实验 A、B 调用你在 `3-1_手写二值形态学.py` 里写的 `erode_binary` 和 `dilate_binary`。作业还没完成时，单元只打印一行“先完成作业里的 …”，OpenCV 的部分照常显示，不会报错；完成作业后重新运行单元即可，不用重启内核。

### 实验 A：结构元实际长什么样，迭代两次等于什么（讲义 3.2、3.3 节，图 3）

先预测：

- `cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))` 里有几个非零像素？它和 3×3 的十字一样吗？
- 一个前景点用 3×3 的十字膨胀两次，得到几个像素？和用 5×5 的十字膨胀一次一样吗？换成 3×3 的方形呢？

下表的数字来自你机器上的 OpenCV，版本不同时可能和讲义里的表不完全一样，以实际打印的为准。

In [ ]:
shapes = {'MORPH_RECT': cv2.MORPH_RECT, 'MORPH_ELLIPSE': cv2.MORPH_ELLIPSE, 'MORPH_CROSS': cv2.MORPH_CROSS}
print(f"{'形状':<16}" + ''.join(f'{k}×{k:<4}' for k in (3, 5, 7, 9)) + ' （非零像素个数）')
for name, code in shapes.items():
    print(f'{name:<18}' + ''.join(f'{int(cv2.getStructuringElement(code, (k, k)).sum()):<6}' for k in (3, 5, 7, 9)))
ellipse3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
cross3 = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
print('3×3 的 ELLIPSE：')
print(ellipse3)
print('它和 3×3 的 CROSS 逐像素相同：', bool(np.array_equal(ellipse3, cross3)))

homework = load_homework('3-1_手写二值形态学.py')
point = np.zeros((11, 11), dtype=np.uint8)
point[5, 5] = 255
square3 = np.ones((3, 3), dtype=np.uint8)
once = attempt(homework.dilate_binary, point, cross3)
if once is not None:
    cross_twice = homework.dilate_binary(once, cross3)
    cross5_once = homework.dilate_binary(point, cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5)))
    square_twice = homework.dilate_binary(homework.dilate_binary(point, square3), square3)
    square5_once = homework.dilate_binary(point, np.ones((5, 5), dtype=np.uint8))
    print('一个点用 3×3 十字膨胀两次：', int(np.count_nonzero(cross_twice)), '个像素；用 5×5 十字膨胀一次：', int(np.count_nonzero(cross5_once)), '个像素')
    print('一个点用 3×3 方形膨胀两次：', int(np.count_nonzero(square_twice)), '个像素；用 5×5 方形膨胀一次：', int(np.count_nonzero(square5_once)), '个像素')
    print('十字膨胀两次的形状：')
    print((cross_twice[2:9, 2:9] > 0).astype(int))
    print('所以“多迭代几次”和“换一个大核”只有方形核可以互换。')

### 实验 B：图像外面算前景还是背景（讲义 3.3 节“边界”，破坏实验 3）

作业的约定是图像外面一律算背景。OpenCV 的默认参数不是这样。

先预测：

- 贴着左上角的 10×10 方块做 3×3 腐蚀：你的 `erode_binary` 剩多少像素？`cv2.erode` 用默认参数剩多少？给 `cv2.erode` 加上 `borderType=cv2.BORDER_CONSTANT, borderValue=0` 之后呢？
- 用你的两个函数直接拼闭运算（先膨胀再腐蚀），作用在一张 20×20 的整幅前景上，会丢多少像素，丢在哪里？

闭运算本来不应该让前景变小。运行之后，按讲义 3.3 节“边界”小节的第 2 条规则自己写一个 `close_fixed(image, kernel)`，目标是这张图上一个像素都不丢。这是讲义 3.12 节的破坏实验 3。

In [ ]:
homework = load_homework('3-1_手写二值形态学.py')
square3 = np.ones((3, 3), dtype=np.uint8)
block = np.zeros((16, 20), dtype=np.uint8)
block[:10, :10] = 255
print('贴着左上角的 10×10 方块做 3×3 腐蚀后剩下的像素：')
print('  cv2.erode 默认参数：', int(np.count_nonzero(cv2.erode(block, square3))))
print('  cv2.erode 指定图像外为背景：', int(np.count_nonzero(cv2.erode(block, square3, borderType=cv2.BORDER_CONSTANT, borderValue=0))))
mine = attempt(homework.erode_binary, block, square3)
if mine is not None:
    print('  你的 erode_binary：', int(np.count_nonzero(mine)))

    def close_naive(image, kernel):
        return homework.erode_binary(homework.dilate_binary(image, kernel), kernel)

    full = np.full((20, 20), 255, dtype=np.uint8)
    closed = close_naive(full, square3)
    print(f'20×20 的整幅前景做 3×3 闭运算：直接拼丢了 {int(np.count_nonzero(full) - np.count_nonzero(closed))} 个像素')
    print('cv2.morphologyEx 默认参数做同样的闭运算丢了', int(np.count_nonzero(full) - np.count_nonzero(cv2.morphologyEx(full, cv2.MORPH_CLOSE, square3))), '个像素')
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.8), layout='constrained')
    for ax, img, title in zip(axes, [full, closed], ['原图：整幅前景', '直接拼的闭运算']):
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.set_xticks([]); ax.set_yticks([])
    plt.show()
    print('下一步：自己写 close_fixed(image, kernel)，让上面这张图一个像素都不丢。')

### 实验 C：开运算和面积过滤各删掉什么（讲义 3.4 节，图 1 和图 4）

一张候选图上有四个真缺陷（1×30 的细划痕、3×30 的粗划痕、3×3 的颗粒、2×2 的颗粒）和两类误报（一条 1 px 宽、25 px 长的竖直边缘线，12 个孤立噪点）。分别做 3×3 方形开运算和“面积小于 5 的连通域删掉”（8 连通）。这个单元只用 OpenCV，不依赖作业。

先预测一张 2 行 6 列的表：两种清理办法下，六样东西各自留下还是被删掉？

- 开运算按什么标准删东西？面积过滤按什么标准？
- 有没有哪一行是全对的（真缺陷都留下，误报都删掉）？
- 把开运算的结构元换成 `np.ones((1, 5), np.uint8)` 再运行一次，哪几格会变？

In [ ]:
candidates = np.zeros((64, 96), dtype=np.uint8)
targets = {
    '1×30 细划痕': (slice(8, 9), slice(10, 40)),
    '3×30 粗划痕': (slice(20, 23), slice(10, 40)),
    '3×3 颗粒': (slice(40, 43), slice(20, 23)),
    '2×2 颗粒': (slice(40, 42), slice(60, 62)),
}
for region in targets.values():
    candidates[region] = 255
edge_line = (slice(30, 55), slice(50, 51))               # 误报：对齐残差沿图形边缘留下的一串超限
candidates[edge_line] = 255
noise_points = [(5, 60), (12, 75), (18, 88), (27, 55), (30, 80), (34, 8), (48, 45), (52, 70), (56, 15), (58, 90), (46, 84), (15, 50)]
for y, x in noise_points:
    candidates[y, x] = 255

def area_filter(image, min_area):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(image, 8)
    keep = np.zeros_like(image)
    for label in range(1, count):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            keep[labels == label] = 255
    return keep

results = {
    '不处理': candidates,
    '3×3 方形开运算': cv2.morphologyEx(candidates, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)),
    '面积过滤（至少 5 个像素）': area_filter(candidates, 5),
}
print(f"{'处理':<18}" + ''.join(f'{name:<12}' for name in targets) + '边缘线（误报）  剩下的噪点')
for name, img in results.items():
    kept = ['留下' if np.any(img[region]) else '删掉' for region in [*targets.values(), edge_line]]
    left = sum(int(img[y, x] > 0) for y, x in noise_points)
    print(f'{name:<20}' + ''.join(f'{k:<14}' for k in kept) + f'{left} 个')

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout='constrained')
for ax, (name, img) in zip(axes, results.items()):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()
print('“留下”指该位置上还有前景像素。对照讲义 3.0 节的表：每种清理都在表达一个假设，什么形状算缺陷，什么形状算噪声。')

## 破坏实验与记录

- 用窄高核和宽扁核分别做闭运算，比较方向先验。
- 增大开运算核，找到主体开始被误删的临界尺寸。
- 找到使右侧两个矩形错误粘连的最小核宽。

把前景像素数、连通域数和失败解释写入 `实验记录.md`。